# Phase 9A — SchemaLinker Stage 1 CoT SFT

Fine-tunes **Qwen3-8B** with QLoRA (4-bit NF4) on `sql_cot_train.json` (6748 entries).  
Checkpoints saved to Google Drive every ~20 min. Re-running this notebook after a disconnect resumes automatically.

**Runtime:** Colab T4 GPU (15.36 GB VRAM)  
**Estimated time:** 2.5–3 hours

## Cell 1 — Verify GPU

In [ ]:
!nvidia-smi

Expected: `Tesla T4, 15360 MiB`. If you see CPU only, go to **Runtime → Change runtime type → T4 GPU**.

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/codegen/checkpoints/sl_sql_stage1'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoint dir ready: {CHECKPOINT_DIR}')

## Cell 3 — Clone or update repo

In [ ]:
%%bash
set -e
cd /content

if [ -d "Codegen/.git" ]; then
    echo "Repo exists — pulling latest changes"
    cd Codegen
    git fetch origin
    git checkout phase/9a-schema-linker-stage1
    git pull origin phase/9a-schema-linker-stage1
else
    echo "Cloning repo"
    git clone https://github.com/kethansplunk/Codegen.git
    cd Codegen
    git checkout phase/9a-schema-linker-stage1
fi

echo "Branch: $(git branch --show-current)"
echo "Latest commit: $(git log --oneline -1)"

## Cell 4 — Install dependencies

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes

## Cell 5 — Verify training data

In [ ]:
import json

DATA_PATH = '/content/Codegen/Data/cot_data/sql_cot_train.json'
with open(DATA_PATH) as f:
    data = json.load(f)

print(f'Training samples : {len(data)}')
print(f'Keys             : {list(data[0].keys())}')
print(f'Sample question  : {data[0]["question"]}')
assert len(data) >= 6000, 'Expected at least 6000 samples'
print('Data check ✅')

## Cell 6 — Run training

- First run: trains from scratch, saves checkpoint every 200 steps (~20 min)
- After a disconnect: re-run this cell — it auto-resumes from the last Drive checkpoint

In [ ]:
%%bash
cd /content/Codegen
python -m src.schema_linker.train_stage1 \
    --data Data/cot_data/sql_cot_train.json \
    --model Qwen/Qwen3-8B \
    --out /content/drive/MyDrive/codegen/checkpoints/sl_sql_stage1

## Cell 7 — Verify adapter output

In [ ]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/codegen/checkpoints/sl_sql_stage1'
files = os.listdir(CHECKPOINT_DIR)
print('Files in output dir:')
for f in sorted(files):
    path = os.path.join(CHECKPOINT_DIR, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  {f:<45} {size_mb:>8.1f} MB')
    else:
        print(f'  {f}/')

adapter = os.path.join(CHECKPOINT_DIR, 'adapter_model.safetensors')
if os.path.exists(adapter):
    size = os.path.getsize(adapter) / 1e6
    print(f'\nAdapter size: {size:.0f} MB  (expected ~260 MB) ✅')
else:
    print('\nadapter_model.safetensors not found — training may still be running or did not complete.')